# Tratamento dos Dados — Hospitais Habilitados em Oncologia

**Fontes:**
- `hospitais-habilitados-em-oncologia-abril-2025.xlsx`
- `hospitais-habilitados-em-reconstrucao-mamaria.xlsx`
- `tratamento-sincronico.xlsx`


---
## 1. Imports

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais

Ajuste aqui os caminhos de entrada/saída e a lista de colunas a remover antes de executar o pipeline.

In [2]:
# ── Caminhos ─────────────────────────────────────────────────────────────────
ENTRADA = Path("/home/anna/tcc/Variáveis/Hospitais Habilitados")
SAIDA   = Path("/home/anna/tcc/Tratamento/Hospitais/tabela_hospitais_habilitados.csv")

# ── Arquivos de entrada ───────────────────────────────────────────────────────
ARQUIVO_ONCOLOGIA    = ENTRADA / "hospitais-habilitados-em-oncologia-abril-2025.xlsx"
ARQUIVO_RECONSTRUCAO = ENTRADA / "hospitais-habilitados-em-reconstrucao-mamaria.xlsx"
ARQUIVO_SINCRONICO   = ENTRADA / "tratamento-sincronico.xlsx"

# ── Tabelas de municípios (tenta a primeira que existir) ──────────────────────
CANDIDATOS_MUNICIPIOS = [
    Path("/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv"),
    Path("/home/anna/tcc/Tratamento/Municípios/municipios_regionais_normalizados.csv"),
]

# ── Colunas a remover da tabela final ─────────────────────────────────────────
# Altere esta lista para incluir ou excluir colunas do resultado
COLUNAS_REMOVER = [
    "UF",
    "CNES",
    "CÓDIGO",
    "CNPJ",
    "GESTÃO",
    "TIPO DE HABILITAÇÃO (CÓDIGO DO CNES)",
    "TIPO DE HABILITAÇÃO (Descrição)",
    "IBGE",
    "MACRORREGIÃO DE SAÚDE",
]

SAIDA.parent.mkdir(parents=True, exist_ok=True)
print(f"✅ Pasta de saída: {SAIDA.parent}")

✅ Pasta de saída: /home/anna/tcc/Tratamento/Hospitais


---
## 3. Funções de Normalização de Texto

Funções para padronizar nomes de municípios antes do join com a tabela de códigos IBGE.
O pipeline de match usa três níveis:
1. `remover_acentos` — remove diacríticos
2. `normalizar_municipio` — strip + lowercase + sem acentos
3. `normalizar_chave_municipio` — normaliza pontuação e resolve aliases conhecidos

In [3]:
def remover_acentos(texto: str) -> str:
    """Remove todos os diacríticos (acentos, cedilha etc.) de uma string."""
    normalizado = unicodedata.normalize("NFD", texto)
    return "".join(c for c in normalizado if unicodedata.category(c) != "Mn")


def normalizar_municipio(valor: object) -> object:
    """Strip + lowercase + sem acentos. Retorna o valor original se não for string."""
    if isinstance(valor, str):
        return remover_acentos(valor.strip()).lower()
    return valor


def normalizar_chave_municipio(valor: object) -> object:
    """
    Normalização completa para uso como chave de join:
    - Aplica normalizar_municipio
    - Substitui pontos e hífens por espaço e colapsa espaços múltiplos
    - Resolve aliases de grafias conhecidas
    """
    if not isinstance(valor, str):
        return valor

    texto = normalizar_municipio(valor)
    texto = " ".join(str(texto).replace(".", " ").replace("-", " ").split())

    aliases = {
        "gov valadares": "governador valadares",
        "sao joao del rei": "sao joao del rei",
    }
    return aliases.get(texto, texto)


✅ Funções de normalização definidas.


---
## 4. Carregamento dos Dados

### 4.1 Arquivos de hospitais habilitados

Três planilhas com estruturas distintas:
- **Oncologia:** cabeçalho na linha 2 (`header=1`), colunas 0-2 com forward-fill (células mescladas no Excel)
- **Reconstrução Mamária:** cabeçalho na linha 1 (`header=0`)
- **Tratamento Sincrônico:** cabeçalho na linha 2 (`header=1`), filtra linhas sem UF na coluna 1

In [4]:
def carregar_dados() -> tuple:
    """Lê as três planilhas brutas e retorna os DataFrames sem filtro de UF."""
    # Oncologia
    df_onco = pd.read_excel(ARQUIVO_ONCOLOGIA, header=1)
    df_onco.columns = df_onco.columns.str.strip()
    colunas_ffill = [df_onco.columns[0], df_onco.columns[1], df_onco.columns[2]]
    df_onco[colunas_ffill] = df_onco[colunas_ffill].ffill()
    df_onco = df_onco.dropna(subset=[df_onco.columns[4]]).reset_index(drop=True)

    # Reconstrução Mamária
    df_recon = pd.read_excel(ARQUIVO_RECONSTRUCAO, header=0)
    df_recon.columns = df_recon.columns.str.strip()

    # Tratamento Sincrônico
    df_sinc = pd.read_excel(ARQUIVO_SINCRONICO, header=1)
    df_sinc.columns = df_sinc.columns.str.strip()
    df_sinc = df_sinc[df_sinc[df_sinc.columns[1]].notna()].reset_index(drop=True)

    return df_onco, df_recon, df_sinc


df_onco, df_recon, df_sinc = carregar_dados()


╔══════════════════════════════════════════════════╗
║         Arquivos carregados (sem filtro)         ║
╠══════════════════════════════════════════════════╣
║  Oncologia            : 370                     ║
║  Reconstrução Mamária : 165                     ║
║  Tratamento Sincrônico: 15                      ║
╚══════════════════════════════════════════════════╝


---
## 5. Análise Inicial dos Dados

Inspeção rápida de cada planilha antes do filtro por UF.

In [5]:
for nome, df in [("Oncologia", df_onco), ("Reconstrução Mamária", df_recon), ("Tratamento Sincrônico", df_sinc)]:
    nulos = int(df.isnull().sum().sum())
    display(Markdown(f"**{nome}** — {df.shape[0]} linhas × {df.shape[1]} colunas | nulos totais: {nulos}"))
    display(df.head(3))

**Oncologia** — 370 linhas × 8 colunas | nulos totais: 56

,UF,MUNICÍPIO,MACRORREGIÃO DE SAÚDE,TOTAL DE HABILITAÇÕES,ESTABELECIMENTO,CNES,CÓDIGO,HABILITAÇÃO
0,AC,Rio Branco,Macro Única - AC,1,Hospital da Fundação Hospitalar Estadual do Acre,2001586,17.07,UNACON com Serviço de Radioterapia
1,AL,Arapiraca,2ª Macrorregião de Saúde,1,Complexo Hospitalar Manoel André - CHAMA,2005417,17.07,UNACON com Serviço de Radioterapia
2,AL,Maceió,1ª Macrorregião de Saúde,1,Hospital da Santa Casa de Misericórdia de Maceió,2007037,17.13,CACON com Serviço de Oncologia Pediátrica


**Reconstrução Mamária** — 165 linhas × 9 colunas | nulos totais: 33

,UF,IBGE,MUNICÍPIO,ESTABELECIMENTO,CNES,CNPJ,GESTÃO,TIPO DE HABILITAÇÃO (CÓDIGO DO CNES),TIPO DE HABILITAÇÃO (Descrição)
0,BA,292740,SALVADOR,HOSPITAL ARISTIDES MALTEZ,0003786,15.180.961/0001-00,MUNICIPAL,17.23,Reconstrução Mamária Pós-Mastectomia Total
1,BA,290000,SALVADOR,HOSPITAL UNIVERSITÁRIO PROFESSOR EDGARD SANTOS,0003816,15.180.714/0002-87,ESTADUAL,17.23,Reconstrução Mamária Pós-Mastectomia Total
2,BA,293135,TEIXEIRA DE FREITAS,HOSPITAL MUNICIPAL DE TEIXEIRA DE FREITAS,2301318,13.650.403/0001-28,MUNICIPAL,17.23,Reconstrução Mamária Pós-Mastectomia Total


**Tratamento Sincrônico** — 15 linhas × 8 colunas | nulos totais: 0

,TOTAL DE HABILITAÇÕES,UF,MUNICÍPIO,MACRORREGIÃO DE SAÚDE,ESTABELECIMENTO,CNES,CÓDIGO,HABILITAÇÃO
0,1,BA,Salvador,Leste,Hospital Aristides Maltez,3786.0,17.22,TRATAMENTOS INTEGRADOS SINCRONICOS EM ONCOLOGIA
1,1,BA,Vitória da Conquista,Sudoeste,Hospital Geral de Vitória da Conquista,2402076.0,17.22,TRATAMENTOS INTEGRADOS SINCRONICOS EM ONCOLOGIA
2,1,CE,Fortaleza,1ª Macro-Fortaleza,Hospital Haroldo Juaçaba,2723220.0,17.22,TRATAMENTOS INTEGRADOS SINCRONICOS EM ONCOLOGIA


---
## 6. Filtro por Estado (MG) e União das Tabelas

Filtra apenas registros de Minas Gerais e une os três DataFrames em uma única tabela,
adicionando a coluna `TIPO` para identificar a origem de cada linha.

In [6]:
def filtrar_mg(df_onco, df_recon, df_sinc) -> tuple:
    """Filtra cada DataFrame mantendo apenas linhas onde a coluna de UF == 'MG'."""
    col_onco  = df_onco.columns[0]
    col_recon = df_recon.columns[0]
    col_sinc  = df_sinc.columns[1]   # Sincrônico tem metadado na col 0; UF está na col 1

    df_onco_mg  = df_onco [df_onco [col_onco ].astype(str).str.strip() == "MG"].reset_index(drop=True)
    df_recon_mg = df_recon[df_recon[col_recon].astype(str).str.strip() == "MG"].reset_index(drop=True)
    df_sinc_mg  = df_sinc [df_sinc [col_sinc ].astype(str).str.strip() == "MG"].reset_index(drop=True)

    return df_onco_mg, df_recon_mg, df_sinc_mg


def unir_tabelas(df_onco_mg, df_recon_mg, df_sinc_mg) -> pd.DataFrame:
    """Concatena os três DataFrames com rótulo de tipo na coluna 'TIPO'."""
    tabela = pd.concat(
        {
            "Oncologia"             : df_onco_mg,
            "Reconstrução Mamária"  : df_recon_mg,
            "Tratamento Sincrônico" : df_sinc_mg,
        },
        axis=0, sort=False,
    )
    tabela.index.names = ["TIPO", "ID_ORIGINAL"]
    return tabela.reset_index(level="TIPO")


df_onco_mg, df_recon_mg, df_sinc_mg = filtrar_mg(df_onco, df_recon, df_sinc)
tabela_unida = unir_tabelas(df_onco_mg, df_recon_mg, df_sinc_mg)

print("╔══════════════════════════════════════════════════╗")
print("║            Após filtro MG                        ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Oncologia            : {len(df_onco_mg):<23} ║")
print(f"║  Reconstrução Mamária : {len(df_recon_mg):<23} ║")
print(f"║  Tratamento Sincrônico: {len(df_sinc_mg):<23} ║")
print(f"║  Total unido          : {len(tabela_unida):<23} ║")
print("╚══════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════╗
║            Após filtro MG                        ║
╠══════════════════════════════════════════════════╣
║  Oncologia            : 40                      ║
║  Reconstrução Mamária : 24                      ║
║  Tratamento Sincrônico: 1                       ║
║  Total unido          : 65                      ║
╚══════════════════════════════════════════════════╝


---
## 7. Enriquecimento com Código IBGE

Carrega a tabela de municípios × código IBGE e faz o join pelo nome normalizado do município.
Municípios sem correspondência são listados para revisão manual.

In [7]:
def carregar_mapa_codigo_municipio() -> dict:
    """
    Carrega o mapeamento municipio_normalizado → codigo_ibge do primeiro arquivo
    de CANDIDATOS_MUNICIPIOS que existir no disco.
    """
    arquivo = next((p for p in CANDIDATOS_MUNICIPIOS if p.exists()), None)
    if arquivo is None:
        candidatos = ", ".join(str(p) for p in CANDIDATOS_MUNICIPIOS)
        raise FileNotFoundError(f"Nenhum arquivo de municípios encontrado. Tentativas: {candidatos}")

    df = pd.read_csv(arquivo, encoding="utf-8-sig")
    obrigatorias = {"codigo_ibge", "municipio_normalizado"}
    faltando = obrigatorias - set(df.columns)
    if faltando:
        raise ValueError(f"Arquivo de municípios sem colunas obrigatórias: {sorted(faltando)}")

    df["municipio_chave"] = df["municipio_normalizado"].map(normalizar_chave_municipio)
    df = df.dropna(subset=["municipio_chave", "codigo_ibge"])

    return dict(zip(df["municipio_chave"].astype(str), df["codigo_ibge"].astype(int)))


def gerar_tabela_habilitados(tabela: pd.DataFrame) -> pd.DataFrame:
    """
    A partir da tabela unida:
    1. Remove colunas de COLUNAS_REMOVER
    2. Normaliza o campo de município
    3. Faz join com o mapa de código IBGE
    4. Reordena colunas: MUNICÍPIOS, CÓDIGO, restante
    """
    # Detecta coluna de município
    if "MUNICÍPIO" in tabela.columns:
        col_municipio = "MUNICÍPIO"
    elif "MUNICÍPIOS" in tabela.columns:
        col_municipio = "MUNICÍPIOS"
    else:
        raise ValueError(f"Coluna de município não encontrada. Colunas disponíveis: {tabela.columns.tolist()}")

    mapa_codigo = carregar_mapa_codigo_municipio()
    df = tabela.copy()

    # Remove colunas desnecessárias
    colunas_existentes = [c for c in COLUNAS_REMOVER if c in df.columns]
    df = df.drop(columns=colunas_existentes)

    # Normaliza e faz join com código IBGE
    df[col_municipio]    = df[col_municipio].map(normalizar_municipio)
    df["municipio_chave"] = df[col_municipio].map(normalizar_chave_municipio)
    df["CÓDIGO"]          = df["municipio_chave"].map(mapa_codigo)
    df = df.drop(columns=["municipio_chave"])

    # Padroniza nome da coluna de município
    if col_municipio != "MUNICÍPIOS":
        df = df.rename(columns={col_municipio: "MUNICÍPIOS"})

    # Converte código para Int64
    df["CÓDIGO"] = pd.to_numeric(df["CÓDIGO"], errors="coerce").astype("Int64")

    # Reordena colunas
    cols_fixas  = ["MUNICÍPIOS", "CÓDIGO"]
    cols_demais = [c for c in df.columns if c not in cols_fixas]
    return df[cols_fixas + cols_demais]


df_tabela = gerar_tabela_habilitados(tabela_unida)

# Relatório de municípios sem código
sem_codigo = df_tabela[df_tabela["CÓDIGO"].isna()]["MUNICÍPIOS"].dropna().unique().tolist()
if sem_codigo:
    display(Markdown(f"⚠️ **{len(sem_codigo)} município(s) sem correspondência de código IBGE:**"))
    for m in sorted(sem_codigo):
        print(f"  - {m}")
else:
    print("Todos os municípios tiveram código IBGE encontrado.")

print(f"\nShape final: {df_tabela.shape}")

✅ Todos os municípios tiveram código IBGE encontrado.

Shape final: (65, 6)


---
## 8. Inspeção Pós-Processamento

Verificação da tabela final antes de salvar: distribuição por tipo, municípios únicos e valores ausentes.

In [8]:
display(Markdown("**Distribuição por tipo de habilitação:**"))
display(
    df_tabela["TIPO"].value_counts().rename_axis("Tipo").reset_index(name="Qtd")
    .style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .hide(axis="index")
)

display(Markdown(f"**Municípios únicos com código IBGE identificado:** {df_tabela['CÓDIGO'].notna().sum()} / {len(df_tabela)}"))

missing = df_tabela.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    display(Markdown("**Valores ausentes por coluna:**"))
    display(missing.to_frame("Nulos").style.background_gradient(cmap="Reds"))
else:
    display(Markdown("✅ **Sem valores ausentes nas colunas principais.**"))

display(Markdown("**Primeiras linhas da tabela final:**"))
display(df_tabela.head(5))

**Distribuição por tipo de habilitação:**

Tipo,Qtd
Oncologia,40
Reconstrução Mamária,24
Tratamento Sincrônico,1


**Municípios únicos com código IBGE identificado:** 65 / 65

**Valores ausentes por coluna:**

,Nulos
TOTAL DE HABILITAÇÕES,29
HABILITAÇÃO,25


**Primeiras linhas da tabela final:**

,MUNICÍPIOS,CÓDIGO,TIPO,TOTAL DE HABILITAÇÕES,ESTABELECIMENTO,HABILITAÇÃO
ID_ORIGINAL,,,,,,
0,belo horizonte,3106200,Oncologia,1,Hospital Luxemburgo/Associação dos Amigos do H...,CACON
1,belo horizonte,3106200,Oncologia,1,Hospital da Baleia/Fundação Benjamin Guimarães,"UNACON com Serviços de Radioterapia, de Hemato..."
2,belo horizonte,3106200,Oncologia,1,Hospital Felício Rocho/Fundação Felice Rosso,UNACON com Serviços de Radioterapia e de Hemat...
3,belo horizonte,3106200,Oncologia,1,Hospital das Clínicas da UFMG,UNACON com Serviços de Hematologia e de Oncolo...
4,belo horizonte,3106200,Oncologia,1,Hospital da Santa Casa de Misericórdia de Belo...,CACON com Serviço de Oncologia Pediátrica


---
## 9. Salvando o Resultado


In [9]:
df_tabela.to_csv(SAIDA, index=False, encoding="utf-8-sig")

╔══════════════════════════════════════════════════════╗
║              Arquivo gerado com sucesso              ║
╠══════════════════════════════════════════════════════╣
║  Pasta de entrada : /home/anna/tcc/Variáveis/Hospitais Habilitados║
║  Arquivo de saída : tabela_hospitais_habilitados.csv ║
║  Total de linhas  : 65                               ║
║  Oncologia (MG)   : 40                               ║
║  Reconstrução (MG): 24                               ║
║  Sincrônico (MG)  : 1                                ║
╚══════════════════════════════════════════════════════╝
